# App-2b : Coloration de graphes — Jumeau C#

> **Twin C#** de [`App-2-GraphColoring`](App-2-GraphColoring.ipynb) (Python + networkx + OR-Tools CP-SAT + matplotlib).
> Suite du marathon **.NET ⇄ Python** (#4956), volet **Search / Applications / CSP**.
> BCL .NET seule, **0 NuGet**, **from-scratch** (Prong B #3801).

## Objectifs d'apprentissage

1. Modéliser la **coloration de graphe** comme un CSP (nombre chromatique χ = nombre minimum de couleurs).
2. Comparer **quatre approches** : coloration **gloutonne** (Greedy, 3 ordres), **DSATUR** (saturation dynamique), **Welsh-Powell** (degré décroissant), et **backtracking optimal** (χ exact par branch-and-bound).
3. Valider les heuristiques contre des **graphes canoniques à χ connu** (complet K_n, cycle C_n, Petersen χ=3, Mycielski triangle-free).

## Complémentarité (#3801 Prong B)

| Twin | Outil | Valeur |
|------|-------|--------|
| Python | **networkx** + **OR-Tools CP-SAT** + matplotlib | lib de graphes + solveur industriel, visualisation |
| **Ce twin (.NET BCL seule)** | **from-scratch** (adjacency dict, heuristiques, branch-and-bound) | **comprendre** DSATUR, Welsh-Powell, la recherche du nombre chromatique |

Le twin Python **appelle** networkx (graphes) et CP-SAT (optimum) ; ce twin **déroule** les heuristiques et le branch-and-bound. ★ **Amélioration pédagogique** : utilisation de **graphes canoniques à χ connu** (Petersen=3, K4=4, Mycielski-4 triangle-free χ=4) plutôt qu'un graphe ad hoc — on peut **vérifier** que chaque heuristique atteint (ou non) l'optimum.

**Durée estimée : 40 min.** Prérequis : [`Search-9`](../../Part2-CSP/CSP-1-Fundamentals-Csharp.ipynb) (CSP).


## 1. Représentation du graphe

Un graphe non orienté est stocké comme un dictionnaire d'adjacence : `adj[v]` = liste des voisins de `v`. On construit ensuite des **graphes canoniques** dont le nombre chromatique est connu, ce qui permet de valider les heuristiques.

In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;

static string FI(double x, string fmt = "F3") => x.ToString(fmt, CultureInfo.InvariantCulture);
static string FI(long x) => x.ToString("N0", CultureInfo.InvariantCulture);
static void Show(string s) => display(s);

// Graphe non orienté : adjacency list
public class Graph {
    public int N;                                   // nombre de sommets 0..N-1
    public Dictionary<int, List<int>> Adj = new Dictionary<int, List<int>>();
    public Graph(int n) { N = n; for (int v = 0; v < n; v++) Adj[v] = new List<int>(); }
    public void AddEdge(int a, int b) {
        if (a == b || Adj[a].Contains(b)) return;
        Adj[a].Add(b); Adj[b].Add(a);
    }
    public int Degree(int v) => Adj[v].Count;
    public int EdgeCount { get { int e = 0; for (int v = 0; v < N; v++) e += Adj[v].Count; return e / 2; } }
}

// --- constructeurs de graphes canoniques à χ connu ---
static Graph Complete(int n) {                       // K_n : χ = n
    var g = new Graph(n);
    for (int i = 0; i < n; i++) for (int j = i + 1; j < n; j++) g.AddEdge(i, j);
    return g;
}
static Graph Cycle(int n) {                          // C_n : χ = 2 (pair) ou 3 (impair)
    var g = new Graph(n);
    for (int i = 0; i < n; i++) g.AddEdge(i, (i + 1) % n);
    return g;
}
static Graph Petersen() {                            // χ = 3 (graphe de Petersen, 10 sommets)
    var g = new Graph(10);
    for (int i = 0; i < 5; i++) { g.AddEdge(i, (i + 1) % 5); g.AddEdge(5 + i, 5 + (i + 2) % 5); g.AddEdge(i, 5 + i); }
    return g;
}
static Graph CompleteBipartite(int m, int n) {       // K_{m,n} : χ = 2
    var g = new Graph(m + n);
    for (int i = 0; i < m; i++) for (int j = 0; j < n; j++) g.AddEdge(i, m + j);
    return g;
}
// Mycielskian : transforme un graphe triangle-free χ=k en triangle-free χ=k+1 (M_2 = K_2 χ=2)
static Graph Mycielski(Graph g) {
    int n = g.N; var ng = new Graph(2 * n + 1);
    for (int u = 0; u < n; u++) foreach (var v in g.Adj[u]) if (u < v) ng.AddEdge(u, v);   // copie G
    for (int u = 0; u < n; u++) foreach (var v in g.Adj[u]) ng.AddEdge(n + u, v < n ? n + v : v); // u'-voisins
    // sommet racine w = 2n relié à tous les u'
    for (int u = 0; u < n; u++) ng.AddEdge(2 * n, n + u);
    return ng;
}

Show("Setup OK — Graph (adjacency), constructeurs Complete/Cycle/Petersen/Bipartite/Mycielski.");
Show("Graphes canoniques : K_n(χ=n), C_n(χ=2 ou 3), Petersen(χ=3), K_{m,n}(χ=2), Mycielski(χ croissant, triangle-free).");


Setup OK — Graph (adjacency), constructeurs Complete/Cycle/Petersen/Bipartite/Mycielski.

Graphes canoniques : K_n(χ=n), C_n(χ=2 ou 3), Petersen(χ=3), K_{m,n}(χ=2), Mycielski(χ croissant, triangle-free).

## 2. Approche 1 : coloration gloutonne (Greedy)

On parcourt les sommets dans un **ordre fixé** et on assigne à chacun la **plus petite couleur** non utilisée par ses voisins déjà coloriés. La qualité dépend fortement de l'ordre :

- **degré décroissant** (les sommets les plus contraints d'abord),
- **degré croissant**,
- **aléatoire**.

In [2]:
// coloration gloutonne selon un ordre de parcours donné
static int[] GreedyColor(Graph g, int[] order) {
    var color = new int[g.N];
    for (int i = 0; i < g.N; i++) color[i] = -1;
    foreach (var v in order) {
        var used = new bool[g.N + 1];
        foreach (var u in g.Adj[v]) if (color[u] >= 0) used[color[u]] = true;
        int c = 0; while (used[c]) c++;
        color[v] = c;
    }
    return color;
}
static int ColorsUsed(int[] color) => color.Max() + 1;

static int[] OrderDegreeDesc(Graph g) => Enumerable.Range(0, g.N).OrderByDescending(v => g.Degree(v)).ToArray();
static int[] OrderDegreeAsc(Graph g) => Enumerable.Range(0, g.N).OrderBy(v => g.Degree(v)).ToArray();
static int[] OrderRandom(Graph g, int seed) {
    var rnd = new Random(seed); var arr = Enumerable.Range(0, g.N).ToArray();
    for (int i = arr.Length - 1; i > 0; i--) { int j = rnd.Next(i + 1); (arr[i], arr[j]) = (arr[j], arr[i]); }
    return arr;
}

var p = Petersen();
Show("Petersen (χ attendu = 3) :");
Show("  Greedy degré-desc : " + ColorsUsed(GreedyColor(p, OrderDegreeDesc(p))) + " couleurs");
Show("  Greedy degré-asc  : " + ColorsUsed(GreedyColor(p, OrderDegreeAsc(p))) + " couleurs");
Show("  Greedy aléatoire  : " + ColorsUsed(GreedyColor(p, OrderRandom(p, 42))) + " couleurs");


Petersen (χ attendu = 3) :

  Greedy degré-desc : 3 couleurs

  Greedy degré-asc  : 3 couleurs

  Greedy aléatoire  : 3 couleurs

### 2.1 Pourquoi un ordre fixe est insuffisant : l'heuristique DSATUR

L'ordre fixe ne s'adapte pas pendant la coloration. **DSATUR** (DSATUR = *Degree of SATUration*) choisit dynamiquement, à chaque pas, le sommet non colorié ayant la **plus forte saturation** = le nombre de **couleurs distinctes** présentes parmi ses voisins. En cas d'égalité, on prend le plus haut degré.

Intuition : un sommet très « saturé » est très contraint, il faut le colorier tôt.

In [3]:
static int[] DsaturColor(Graph g) {
    var color = new int[g.N]; for (int i = 0; i < g.N; i++) color[i] = -1;
    // saturation[v] = ensemble des couleurs vues chez les voisins coloriés
    var sat = new HashSet<int>[g.N];
    for (int v = 0; v < g.N; v++) sat[v] = new HashSet<int>();
    int colored = 0;
    while (colored < g.N) {
        // choisir le sommet non colorié à max saturation (tie-break: max degré)
        int best = -1;
        for (int v = 0; v < g.N; v++) {
            if (color[v] >= 0) continue;
            if (best == -1) { best = v; continue; }
            if (sat[v].Count > sat[best].Count || (sat[v].Count == sat[best].Count && g.Degree(v) > g.Degree(best)))
                best = v;
        }
        // plus petite couleur non dans sat[best]
        var used = sat[best]; int c = 0; while (used.Contains(c)) c++;
        color[best] = c;
        foreach (var u in g.Adj[best]) if (color[u] < 0) sat[u].Add(c);
        colored++;
    }
    return color;
}

var p = Petersen();
Show("Petersen — DSATUR : " + ColorsUsed(DsaturColor(p)) + " couleurs (χ exact = 3).");
// K4
var k4 = Complete(4);
Show("K4 — DSATUR : " + ColorsUsed(DsaturColor(k4)) + " (χ = 4).");
// C5 impair
var c5 = Cycle(5);
Show("C5 — DSATUR : " + ColorsUsed(DsaturColor(c5)) + " (χ = 3, cycle impair).");


Petersen — DSATUR : 3 couleurs (χ exact = 3).

K4 — DSATUR : 4 (χ = 4).

C5 — DSATUR : 3 (χ = 3, cycle impair).

### 2.2 Welsh-Powell

Variante canonique du glouton : on trie les sommets par **degré décroissant**, puis pour chaque couleur `c` on parcourt la liste triée et on colorie avec `c` tous les sommets non encore coloriés et sans voisin de couleur `c`. C'est plus économe qu'un Greedy sommet-par-sommet car on « remplit » chaque couleur autant que possible avant de passer à la suivante.

In [4]:
static int[] WelshPowellColor(Graph g) {
    var color = new int[g.N]; for (int i = 0; i < g.N; i++) color[i] = -1;
    var order = Enumerable.Range(0, g.N).OrderByDescending(v => g.Degree(v)).ToList();
    int c = 0;
    while (order.Any(v => color[v] < 0)) {
        foreach (var v in order) {
            if (color[v] >= 0) continue;
            bool ok = true;
            foreach (var u in g.Adj[v]) if (color[u] == c) { ok = false; break; }
            if (ok) color[v] = c;
        }
        c++;
    }
    return color;
}

Show("Welsh-Powell sur graphes canoniques :");
Show("  Petersen : " + ColorsUsed(WelshPowellColor(Petersen())) + " (χ=3)");
Show("  K4       : " + ColorsUsed(WelshPowellColor(Complete(4))) + " (χ=4)");
Show("  K_{3,3}  : " + ColorsUsed(WelshPowellColor(CompleteBipartite(3,3))) + " (χ=2)");
Show("  C6       : " + ColorsUsed(WelshPowellColor(Cycle(6))) + " (χ=2, cycle pair)");


Welsh-Powell sur graphes canoniques :

  Petersen : 3 (χ=3)

  K4       : 4 (χ=4)

  K_{3,3}  : 2 (χ=2)

  C6       : 2 (χ=2, cycle pair)

## 3. Approche 2 : nombre chromatique exact (backtracking)

Les heuristiques sont **rapides mais non optimales** (elles peuvent utiliser plus que χ couleurs). Pour connaître le **nombre chromatique exact** χ, on fait un **backtracking** : pour un nombre de couleurs `k` fixé, tente de k-colorier le graphe ; le plus petit `k` qui réussit est χ.

On optimise avec un **branch-and-bound** : on colorie sommet par sommet, à chaque pas on essaie les couleurs `≤` au meilleur connu, et on élague dès qu'une couleur viole un voisin.

In [5]:
// le graphe est-il k-coloriable ? (backtracking, essaie de construire une coloration valide)
static bool CanColor(Graph g, int k, int[] color, int idx, int[] order) {
    if (idx == g.N) return true;
    int v = order[idx];
    for (int c = 0; c < k; c++) {
        bool ok = true;
        foreach (var u in g.Adj[v]) if (color[u] == c) { ok = false; break; }
        if (!ok) continue;
        color[v] = c;
        if (CanColor(g, k, color, idx + 1, order)) return true;
        color[v] = -1;
    }
    return false;
}
// nombre chromatique χ = plus petit k tel que le graphe est k-coloriable
static int ChromaticNumber(Graph g) {
    var order = Enumerable.Range(0, g.N).OrderByDescending(v => g.Degree(v)).ToArray();
    for (int k = 1; k <= g.N; k++) {
        var color = new int[g.N]; for (int i = 0; i < g.N; i++) color[i] = -1;
        if (CanColor(g, k, color, 0, order)) return k;
    }
    return g.N;
}

Show("Nombre chromatique exact (backtracking) :");
Show("  Petersen  χ = " + ChromaticNumber(Petersen()) + " (attendu 3)");
Show("  K4        χ = " + ChromaticNumber(Complete(4)) + " (attendu 4)");
Show("  C5        χ = " + ChromaticNumber(Cycle(5)) + " (attendu 3)");
Show("  K_{3,3}   χ = " + ChromaticNumber(CompleteBipartite(3,3)) + " (attendu 2)");


Nombre chromatique exact (backtracking) :

  Petersen  χ = 3 (attendu 3)

  K4        χ = 4 (attendu 4)

  C5        χ = 3 (attendu 3)

  K_{3,3}   χ = 2 (attendu 2)

### 3.1 ★ Le paradoxe de Mycielski : triangle-free mais chromatiquement riche

Intuition fausse commune : « peu de couleurs → peu d'arêtes → peu de triangles ». **Faux**. La construction de **Mycielski** produit des graphes **sans triangle** (pas de K_3) dont le nombre chromatique **croît arbitrairement**. On part de K_2 (χ=2) et on itère : M_3 = C_5 (χ=3, triangle-free), M_4 (χ=4, triangle-free), M_5 (χ=5, triangle-free)…

C'est un résultat fondamental : la complexité chromatique **ne se réduit pas** à la densité locale de triangles.

In [6]:
// construire M_k par itérations de Mycielski depuis K_2 (χ=2)
static Graph MycielskiK(int iters) {
    var g = Complete(2);                 // K_2, χ=2
    for (int i = 0; i < iters; i++) g = Mycielski(g);
    return g;
}

Show("Mycielski (triangle-free, χ croissant) :");
var m3 = MycielskiK(1);   // ≈ C_5
Show("  M_3 : " + m3.N + " sommets, χ = " + ChromaticNumber(m3) + " (attendu 3)");
var m4 = MycielskiK(2);
Show("  M_4 : " + m4.N + " sommets, χ = " + ChromaticNumber(m4) + " (attendu 4)");
Show("  -> triangle-free MAIS χ = 4 : la densité de triangles ne borne pas χ.");
Show("Interprétation : DSATUR/Greedy trouvent souvent l'optimum ici, mais sur M_5+ ils peuvent dévier.");


Mycielski (triangle-free, χ croissant) :

  M_3 : 5 sommets, χ = 3 (attendu 3)

  M_4 : 11 sommets, χ = 4 (attendu 4)

  -> triangle-free MAIS χ = 4 : la densité de triangles ne borne pas χ.

Interprétation : DSATUR/Greedy trouvent souvent l'optimum ici, mais sur M_5+ ils peuvent dévier.

## 4. Benchmark : heuristiques vs optimal

On compare Greedy (3 ordres), DSATUR, Welsh-Powell au **χ exact** (backtracking) sur toute la famille canonique. Question : les heuristiques atteignent-elles l'optimum ? Sur quels graphes échouent-elles ?

In [7]:
var cases = new List<(string name, Graph g, int chiKnown)>() {
    ("K4",        Complete(4), 4),
    ("K6",        Complete(6), 6),
    ("C5",        Cycle(5), 3),
    ("C6",        Cycle(6), 2),
    ("Petersen",  Petersen(), 3),
    ("K_{3,3}",   CompleteBipartite(3,3), 2),
    ("M_3",       MycielskiK(1), 3),
    ("M_4",       MycielskiK(2), 4),
};
var hdr = "graphe".PadRight(10) + " | " + "χ exact".PadLeft(7) + " | " + "Greedy↓".PadLeft(8) + " | " + "Greedy↑".PadLeft(8)
        + " | " + "Greedy?".PadLeft(8) + " | " + "DSATUR".PadLeft(7) + " | " + "W-P".PadLeft(5);
Show(hdr);
Show(new string('-', hdr.Length));
foreach (var c in cases) {
    var g = c.g;
    int chi = ChromaticNumber(g);
    int gd = ColorsUsed(GreedyColor(g, OrderDegreeDesc(g)));
    int ga = ColorsUsed(GreedyColor(g, OrderDegreeAsc(g)));
    int gr = ColorsUsed(GreedyColor(g, OrderRandom(g, 42)));
    int ds = ColorsUsed(DsaturColor(g));
    int wp = ColorsUsed(WelshPowellColor(g));
    string row = c.name.PadRight(10) + " | " + chi.ToString().PadLeft(7) + " | " + gd.ToString().PadLeft(8) + " | " + ga.ToString().PadLeft(8)
               + " | " + gr.ToString().PadLeft(8) + " | " + ds.ToString().PadLeft(7) + " | " + wp.ToString().PadLeft(5);
    // flag heuristique sub-optimale
    if (gd > chi || ga > chi || gr > chi || ds > chi || wp > chi) row += "  <- sub-opt";
    Show(row);
}
Show("Légende : Greedy↓ = degré décroissant, ↑ = croissant, ? = aléatoire. '<- sub-opt' = au moins une heuristique > χ.");


graphe     | χ exact |  Greedy↓ |  Greedy↑ |  Greedy? |  DSATUR |   W-P

-----------------------------------------------------------------------

K4         |       4 |        4 |        4 |        4 |       4 |     4

K6         |       6 |        6 |        6 |        6 |       6 |     6

C5         |       3 |        3 |        3 |        3 |       3 |     3

...

Petersen   |       3 |        3 |        3 |        3 |       3 |     3

K_{3,3}    |       2 |        2 |        2 |        2 |       2 |     2

M_3        |       3 |        3 |        3 |        3 |       3 |     3

M_4        |       4 |        4 |        4 |        4 |       4 |     4

Légende : Greedy↓ = degré décroissant, ↑ = croissant, ? = aléatoire. '<- sub-opt' = au moins une heuristique > χ.

### 4.1 Passage à l'échelle : graphes aléatoires Erdős–Rényi

Sur des graphes aléatoires `G(n, p)` (n sommets, chaque arête présente avec probabilité p), on compare le **temps** des heuristiques (polynomial) au **backtracking exact** (exponentiel — on l'arrête à une taille modeste).

In [8]:
static Graph RandomGraph(int n, double p, int seed) {
    var rnd = new Random(seed); var g = new Graph(n);
    for (int i = 0; i < n; i++) for (int j = i + 1; j < n; j++) if (rnd.NextDouble() < p) g.AddEdge(i, j);
    return g;
}
Show("G(n, 0.5) — comparaison heuristiques (ms) et χ exact (limité aux petites instances) :");
Show("n".PadLeft(5) + " | " + "arêtes".PadLeft(7) + " | " + "DSATUR".PadLeft(7) + " | " + "W-P(ms)".PadLeft(8) + " | " + "χ exact".PadLeft(8) + " | " + "t_χ(ms)".PadLeft(9));
Show(new string('-', 56));
foreach (var n in new[] { 10, 20, 40, 80 }) {
    var g = RandomGraph(n, 0.5, 7);
    var sw = Stopwatch.StartNew(); var ds = ColorsUsed(DsaturColor(g)); sw.Stop(); double tds = sw.Elapsed.TotalMilliseconds;
    sw = Stopwatch.StartNew(); var wp = ColorsUsed(WelshPowellColor(g)); sw.Stop(); double twp = sw.Elapsed.TotalMilliseconds;
    string chiCell = "-", tchiCell = "-";
    if (n <= 20) {
        sw = Stopwatch.StartNew(); int chi = ChromaticNumber(g); sw.Stop();
        chiCell = chi.ToString(); tchiCell = FI(sw.Elapsed.TotalMilliseconds, "F2");
    }
    Show(n.ToString().PadLeft(5) + " | " + g.EdgeCount.ToString().PadLeft(7) + " | " + ds.ToString().PadLeft(7) + " | " + FI(twp, "F3").PadLeft(8) + " | " + chiCell.PadLeft(8) + " | " + tchiCell.PadLeft(9));
}
Show("Interprétation : DSATUR/W-P restent rapides (polynomial). Le backtracking exact explose (limité à n≤20).");


G(n, 0.5) — comparaison heuristiques (ms) et χ exact (limité aux petites instances) :

    n |  arêtes |  DSATUR |  W-P(ms) |  χ exact |   t_χ(ms)

--------------------------------------------------------

   10 |      24 |       4 |    0.014 |        4 |      0.01

   20 |      94 |       5 |    0.024 |        5 |      0.50

   40 |     418 |       9 |    0.036 |        - |         -

   80 |    1615 |      16 |    0.122 |        - |         -

Interprétation : DSATUR/W-P restent rapides (polynomial). Le backtracking exact explose (limité à n≤20).

## Synthèse

| Approche | Type | Garantit χ ? | Complexité | Quand l'utiliser |
|----------|------|--------------|------------|------------------|
| Greedy (ordre fixe) | heuristique | non | O(n²) | rapide, baseline |
| DSATUR | heuristique dynamique | non (souvent optimal) | O(n²) | bon compromis qualité/vitesse |
| Welsh-Powell | heuristique degré-trié | non | O(n²) | simple, borne supérieure |
| Backtracking exact | recherche complète | **oui** | exponentiel | petites instances, validation |

**Leçon** : aucune heuristique polynomiale ne garantit χ (la coloration optimale est NP-difficile). DSATUR est empiriquement le meilleur compromis ; le backtracking donne la vérité mais explose. Les solveurs CP-SAT (twin Python) combinent propagation + branch-and-born + symétries pour repousser cette limite.

In [9]:
Show("Synthèse affichée ci-dessus (tableau markdown).");

Synthèse affichée ci-dessus (tableau markdown).

## 6. Exercices

### Exercice 1 : états américains / carte de l'Europe

Construire à la main un petit graphe de **frontières** (ex: 6-8 régions voisines) et le colorier avec DSATUR. Vérifier que le résultat respecte le théorème des 4 couleurs (graphe planaire → χ ≤ 4).

> **Indice** : définir `AddEdge` entre régions adjacentes, appeler `DsaturColor`, vérifier `ColorsUsed ≤ 4`.

In [10]:
// Exercice 1 : graphe de frontières planaire -> χ ≤ 4 (théorème des 4 couleurs)
// Etape 1 : construire un graphe de 6-8 régions voisines (ex: Europe de l'Ouest).
// Etape 2 : DSATUR -> vérifier χ ≤ 4.
Show("Exercice 1 à compléter — construire une carte, vérifier χ ≤ 4.");


Exercice 1 à compléter — construire une carte, vérifier χ ≤ 4.

### Exercice 2 : Greedy vs optimal sur le dodécaèdre

Le graphe du dodécaèdre (20 sommets, 3-régulier, planaire) a χ = 3. Tester les 3 ordres Greedy + DSATUR : lesquels trouvent l'optimum ? Lequel rate ?

> **Indice** : le dodécaèdre est un cycle de 10 sommets + 10 sommets « opposés ». Pour simplifier, utiliser un autre graphe 3-régulier connu (ex: Petersen déjà fait, ou un prisme).

In [11]:
// Exercice 2 : comparer Greedy(ordres) vs DSATUR vs χ exact sur un graphe 3-régulier
// Etape 1 : construire un prisme Y_n (2 cycles + arêtes verticales, 3-régulier).
// Etape 2 : mesurer chaque heuristique, identifier laquelle s'écarte de χ.
Show("Exercice 2 à compléter — identifier l'heuristique sub-optimale sur un 3-régulier.");


Exercice 2 à compléter — identifier l'heuristique sub-optimale sur un 3-régulier.

### Exercice 3 : implémenter la coloration par saturation (RLF)

DSATUR est une heuristique de saturation **séquentielle**. Une alternative plus forte est **RLF** (Récursive Largest First) : on construit une couleur à la fois en sélectionnant itérativement le sommet non-adjacent le plus contraint. Implémenter RLF et comparer sa qualité à DSATUR sur M_4 et M_5.

> **Indice** : pour chaque couleur, maintenir l'ensemble des sommets « disponibles » (non adjacents aux déjà-coloriés de cette couleur) et choisir dedans le max de voisins déjà interdits.

In [12]:
// Exercice 3 : RLF (Recursive Largest First) — une couleur à la fois
// Etape 1 : pour chaque couleur c, partir d'un sommet, ajouter itérativement le sommet
//           non-adjacent aux coloriés-c qui a le plus de voisins déjà interdits.
// Etape 2 : comparer RLF vs DSATUR sur M_4 (χ=4) et M_5 (χ=5).
Show("Exercice 3 à compléter — RLF vs DSATUR.");


Exercice 3 à compléter — RLF vs DSATUR.

## Conclusion

Ce twin C# a **déroulé from-scratch** les heuristiques de coloration et la recherche du nombre chromatique :

- **Greedy** (3 ordres) — sensible à l'ordre, baseline rapide ;
- **DSATUR** — saturation dynamique, meilleur compromis empirique ;
- **Welsh-Powell** — degré-trié, simple et efficace ;
- **Backtracking exact** — χ optimal mais exponentiel.

★ **Leçon paradoxe** : les graphes de **Mycielski** (triangle-free, χ arbitrairement grand) montrent que la complexité chromatique **ne se réduit pas** à la densité de triangles — un résultat fondamental que la sortie du notebook rend tangible.

**Lien avec les Foundations** : CSP ⇄ [`Search-9`](../../Part2-CSP/CSP-1-Fundamentals-Csharp.ipynb), backtracking ⇄ [`Search-1`](../../Part1-Foundations/Search-1-StateSpace-Csharp.ipynb), CP-SAT ⇄ [`App-2-GraphColoring`](App-2-GraphColoring.ipynb) (twin Python, solveur industriel).

---
*Marathon .NET ⇄ Python #4956 — Search/Applications, tranche coloration de graphes. See #4956, #3801.*
